# 02 — Feature Engineering

> **Live product:** company MiniLM vectors for matching are built by `scripts/build_company_embeddings.py` → `app/app_data/company_embeddings.npz`. This notebook remains useful for exploratory feature stores and synthetic leavers.

**Maps from previous project:** Notebook 02 (features + sentence-transformer vectors)

**Input:** `data/processed/companies_master.pkl`

**Tasks:**
1. Engineer company numeric / categorical features
2. Build synthetic leaver profiles (for offline demo & clustering)
3. Encode `profile_text` with sentence-transformers

**Output:** `data/processed/company_feature_store.pkl`, `leaver_feature_store.pkl`  
**Production equivalent:** `scripts/build_company_embeddings.py`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from glos_recommender.matching import build_leaver_profile

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODEL_DIR = PROJECT_ROOT / "models" / "local_minilm_model"

companies = pd.read_pickle(PROCESSED_DIR / "companies_master.pkl")
print(f"Loaded companies: {companies.shape}")

c:\Users\MSI Katana Gaming\HigherED_ML_app\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded companies: (1139, 23)


## 1. Company features

In [2]:
hiring_map = {"low": 0.3, "medium": 0.6, "high": 1.0}
size_map = {"sme": 0.3, "medium": 0.5, "mixed": 0.6, "large": 1.0}

companies["hiring_score"] = companies["hiring_signal"].map(hiring_map).fillna(0.3)
companies["size_score"] = companies["size_band"].map(size_map).fillna(0.5)
companies["priority_score"] = companies["priority_employer"].astype(float)

# Ensure profile_text exists for embedding
if "profile_text" not in companies.columns:
    companies["profile_text"] = (
        companies["name"] + " " + companies["summary"].fillna("") + " " + companies["sectors"].fillna("")
    )

print(companies[["name", "hiring_score", "size_score", "n_opportunities"]].head())

                     name  hiring_score  size_score  n_opportunities
0                Renishaw           1.0         1.0                2
1            Spirax Sarco           1.0         1.0                1
2            GE Aerospace           1.0         1.0                1
3                    Moog           1.0         1.0                1
4  Safran Landing Systems           0.6         1.0                0


## 2. Synthetic leaver cohort

Creates diverse demo leavers so Notebooks 03–04 can cluster and recommend without live form traffic.

In [3]:
SYNTHETIC_FORMS = [
    {
        "leaver_id": "L001",
        "leaver_type": "School leaver (Year 11 / 13)",
        "location": "Cheltenham",
        "courses": ["Computer Science / IT", "Maths / Further Maths"],
        "interests": ["Cybersecurity & digital defence", "Software / apps / web"],
        "passions": ["Solving puzzles / problems", "Protecting systems / people"],
        "work_experience": ["Coding / personal projects"],
        "qualification_level": "A-levels / BTEC / T Levels / Level 3",
        "availability": "Within 3 months",
        "psych_answers": {
            "Q1_problem_style": "analyse", "Q2_energy_source": "discovered",
            "Q3_work_setting": "office_hybrid", "Q4_structure_vs_autonomy": "mix",
            "Q5_risk_novelty": "pioneer", "Q6_team_role": "specialist",
        },
    },
    {
        "leaver_id": "L002",
        "leaver_type": "School leaver (Year 11 / 13)",
        "location": "Tewkesbury / North Gloucestershire",
        "courses": ["Engineering / Design Technology", "Physics / Chemistry / Biology"],
        "interests": ["Aerospace & advanced manufacturing", "Mechanical / electrical engineering"],
        "passions": ["Building or fixing things", "Making products people use"],
        "work_experience": ["Work experience placement (school)"],
        "qualification_level": "A-levels / BTEC / T Levels / Level 3",
        "availability": "Within 6–12 months",
        "psych_answers": {
            "Q1_problem_style": "build", "Q2_energy_source": "shipped",
            "Q3_work_setting": "workshop_lab", "Q4_structure_vs_autonomy": "clear_structure",
            "Q5_risk_novelty": "apply", "Q6_team_role": "specialist",
        },
    },
    {
        "leaver_id": "L003",
        "leaver_type": "University undergraduate (final year / recent graduate)",
        "location": "Cirencester / Cotswolds",
        "courses": ["Agriculture / Food / Environmental"],
        "interests": ["Agri-tech & food systems", "Environment & sustainability"],
        "passions": ["Working outdoors", "Growing food / nature"],
        "work_experience": ["Internship / sandwich year", "Family / farm / small business"],
        "qualification_level": "Undergraduate degree / Level 6",
        "availability": "Within 3 months",
        "psych_answers": {
            "Q1_problem_style": "build", "Q2_energy_source": "helped",
            "Q3_work_setting": "workshop_lab", "Q4_structure_vs_autonomy": "mix",
            "Q5_risk_novelty": "apply", "Q6_team_role": "connector",
        },
    },
    {
        "leaver_id": "L004",
        "leaver_type": "College / FE leaver",
        "location": "Gloucester",
        "courses": ["Health & Social Care"],
        "interests": ["Healthcare & wellbeing", "Public service & community"],
        "passions": ["Helping people face-to-face"],
        "work_experience": ["Volunteering", "Caring responsibilities"],
        "qualification_level": "A-levels / BTEC / T Levels / Level 3",
        "availability": "Immediate",
        "psych_answers": {
            "Q1_problem_style": "people", "Q2_energy_source": "helped",
            "Q3_work_setting": "community", "Q4_structure_vs_autonomy": "clear_structure",
            "Q5_risk_novelty": "careful", "Q6_team_role": "connector",
        },
    },
    {
        "leaver_id": "L005",
        "leaver_type": "University undergraduate (final year / recent graduate)",
        "location": "Cheltenham",
        "courses": ["Business / Management / Marketing", "Creative Arts / Media"],
        "interests": ["Creative / design / events", "Business / sales / entrepreneurship"],
        "passions": ["Creating visual / written content", "Leading teams / organising"],
        "work_experience": ["Part-time retail / hospitality", "Internship / sandwich year"],
        "qualification_level": "Undergraduate degree / Level 6",
        "availability": "Within 3 months",
        "psych_answers": {
            "Q1_problem_style": "create", "Q2_energy_source": "influenced",
            "Q3_work_setting": "studio_events", "Q4_structure_vs_autonomy": "open_ended",
            "Q5_risk_novelty": "pioneer", "Q6_team_role": "ideas",
        },
    },
    {
        "leaver_id": "L006",
        "leaver_type": "School leaver (Year 11 / 13)",
        "location": "Stroud",
        "courses": ["Geography / Environmental Science", "Business / Economics"],
        "interests": ["Environment & sustainability", "Construction & built environment"],
        "passions": ["Working outdoors", "Improving how organisations run"],
        "work_experience": ["Volunteering"],
        "qualification_level": "A-levels / BTEC / T Levels / Level 3",
        "availability": "Within 6–12 months",
        "psych_answers": {
            "Q1_problem_style": "analyse", "Q2_energy_source": "shipped",
            "Q3_work_setting": "workshop_lab", "Q4_structure_vs_autonomy": "mix",
            "Q5_risk_novelty": "apply", "Q6_team_role": "organiser",
        },
    },
    {
        "leaver_id": "L007",
        "leaver_type": "University undergraduate (final year / recent graduate)",
        "location": "Cheltenham",
        "courses": ["Computer Science / Cyber / Software", "Maths / Data / Stats"],
        "interests": ["Data & AI", "Cybersecurity & digital defence"],
        "passions": ["Analysing numbers / patterns", "Solving puzzles / problems"],
        "work_experience": ["Internship / sandwich year", "Coding / personal projects"],
        "qualification_level": "Undergraduate degree / Level 6",
        "availability": "Immediate",
        "psych_answers": {
            "Q1_problem_style": "analyse", "Q2_energy_source": "discovered",
            "Q3_work_setting": "office_hybrid", "Q4_structure_vs_autonomy": "open_ended",
            "Q5_risk_novelty": "pioneer", "Q6_team_role": "specialist",
        },
    },
    {
        "leaver_id": "L008",
        "leaver_type": "College / FE leaver",
        "location": "Gloucester",
        "courses": ["Hospitality / Catering", "Business / Economics"],
        "interests": ["Hospitality / tourism / retail", "Business / sales / entrepreneurship"],
        "passions": ["Helping people face-to-face", "Leading teams / organising"],
        "work_experience": ["Part-time retail / hospitality"],
        "qualification_level": "A-levels / BTEC / T Levels / Level 3",
        "availability": "Immediate",
        "psych_answers": {
            "Q1_problem_style": "people", "Q2_energy_source": "influenced",
            "Q3_work_setting": "community", "Q4_structure_vs_autonomy": "clear_structure",
            "Q5_risk_novelty": "careful", "Q6_team_role": "organiser",
        },
    },
]

leaver_rows = []
for form in SYNTHETIC_FORMS:
    profile = build_leaver_profile(form)
    leaver_rows.append({
        "leaver_id": form["leaver_id"],
        **{k: profile[k] for k in [
            "leaver_type", "location", "courses", "interests", "passions",
            "work_experience", "qualification_level", "availability",
            "profile_text",
        ]},
        "interest_sectors": "|".join(sorted(profile["interest_sectors"])),
        "psych_sectors": "|".join(sorted(profile["psych_sectors"])),
        "entry_routes": "|".join(sorted(profile["entry_routes"])),
        "dominant_riasec": "|".join(profile["psych"]["dominant_riasec"]),
        "role_prefs": "|".join(sorted(profile["psych"]["role_prefs"])),
        "form": form,
    })

leavers = pd.DataFrame(leaver_rows)
print(f"Synthetic leavers: {len(leavers)}")
leavers[["leaver_id", "leaver_type", "interest_sectors", "dominant_riasec"]]

Synthetic leavers: 8


,leaver_id,leaver_type,interest_sectors,dominant_riasec
0,L001,School leaver (Year 11 / 13),cyber_digital,Investigative|Conventional
1,L002,School leaver (Year 11 / 13),aerospace_manufacturing|construction_green,Realistic|Conventional
2,L003,University undergraduate (final year / recent ...,agri_tech_food|construction_green|public_sector,Realistic|Social
3,L004,College / FE leaver,health_care|public_sector,Social|Conventional
4,L005,University undergraduate (final year / recent ...,business_professional|creative_events|hospital...,Artistic|Enterprising
5,L006,School leaver (Year 11 / 13),agri_tech_food|construction_green|public_sector,Conventional|Realistic
6,L007,University undergraduate (final year / recent ...,business_professional|cyber_digital,Investigative|Conventional
7,L008,College / FE leaver,business_professional|hospitality_retail,Conventional|Social


## 3. Sentence-transformer embeddings

In [4]:
if (MODEL_DIR / "config.json").exists():
    model = SentenceTransformer(str(MODEL_DIR))
else:
    model = SentenceTransformer("all-MiniLM-L6-v2")

print("Encoding company profiles...")
company_vectors = model.encode(companies["profile_text"].tolist(), show_progress_bar=True, batch_size=32)
companies["vector"] = list(company_vectors)

print("Encoding leaver profiles...")
leaver_vectors = model.encode(leavers["profile_text"].tolist(), show_progress_bar=True, batch_size=32)
leavers["vector"] = list(leaver_vectors)

print(f"Company vector shape: {company_vectors.shape}")
print(f"Leaver vector shape:  {leaver_vectors.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6631.87it/s]


Encoding company profiles...


Batches: 100%|██████████| 36/36 [00:06<00:00,  5.22it/s]


Encoding leaver profiles...


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.44it/s]

Company vector shape: (1139, 384)
Leaver vector shape:  (8, 384)


## 4. Save feature stores

In [5]:
companies.to_pickle(PROCESSED_DIR / "company_feature_store.pkl")
leavers.to_pickle(PROCESSED_DIR / "leaver_feature_store.pkl")

print("Saved company_feature_store.pkl")
print("Saved leaver_feature_store.pkl")
print("\nProceed to Notebook 03 — Clustering.")

Saved company_feature_store.pkl
Saved leaver_feature_store.pkl

Proceed to Notebook 03 — Clustering.
